# Detección de drones: MFCC con y sin atenuación del fondo

Compara **MFCC + SVM** frente a **atenuación espectral suave + MFCC + SVM** en las carpetas `hexa`, `Quad` y `no_dron`. Las dos primeras son positivas durante todo el archivo; `no_dron` es negativa. Se incluye `154519` completo.

El filtro estima un perfil espectral únicamente con los negativos de entrenamiento de cada partición. Nunca utiliza los negativos de evaluación. La transformación es idéntica para ambas clases. Se conservan 64 filtros Mel, compresión logarítmica y 19 MFCC sin C0, resumidos en media y desviación (38 características).

Es una **máscara espectral inspirada en Wiener**, con estimación simplificada de potencia útil; no una implementación óptima de Wiener ni una garantía de mejora. Los resultados se comparan con idénticas particiones y parámetros del SVM. No se ajustan filtro ni umbral mirando la evaluación.

Ejecuta todas las celdas en orden. Los WAV originales no se modifican. El notebook anterior se conserva en una copia de respaldo. Las salidas antiguas no se reutilizan como resultados de este experimento.


In [ ]:
# Si faltan paquetes, descomenta y ejecuta una vez:
# %pip install numpy scikit-learn matplotlib joblib ipython


In [ ]:
from pathlib import Path
import wave, hashlib, json, csv, platform
import numpy as np

BASE=Path(r'C:\Users\alche\OneDrive\Desktop\Work\repodron\New folder')
FOLDERS=[('hexa',1),('Quad',1),('no_dron',0)]
OUT=BASE/'resultados_filtro_ruido'
CHANNEL=0
WINDOW_S=2.0
PROFILE_STEP_S=4.0  # Muestreo uniforme para el perfil por grabación negativa.
SEED=42
ALPHA=1.0          # Potencia del perfil de ruido; fijada antes de evaluar.
GAIN_FLOOR=0.20    # Piso de ganancia de amplitud; evita anular componentes débiles.
assert ALPHA>=0 and 0<GAIN_FLOOR<=1


## 1. Lectura y auditoría
Excluimos duplicados exactos por contenido PCM y formato. Si un duplicado tiene etiquetas contradictorias, se detiene el proceso. Se conserva el primer archivo en orden de búsqueda. El canal 0 se procesa por separado, sin promediar el estéreo. Se exige una frecuencia común y PCM de 16 bits. La saturación se informa; el filtro no reconstruye lo perdido por clipping.


In [ ]:
def read_audio(path):
    with wave.open(str(path),'rb') as f:
        sr,ch,width=f.getframerate(),f.getnchannels(),f.getsampwidth()
        assert width==2 and f.getcomptype()=='NONE',f'Formato no compatible: {path}'
        raw=f.readframes(f.getnframes())
    assert CHANNEL<ch
    x=np.frombuffer(raw,dtype='<i2').reshape(-1,ch)[:,CHANNEL].astype(np.float64)/32768
    digest=hashlib.sha256(raw).hexdigest()
    return x,sr,ch,(sr,ch,width,digest)

records=[]; duplicates=[];seen={}
for folder,label in FOLDERS:
    root=BASE/folder
    assert root.is_dir(),f'Carpeta ausente: {root}'
    for path in sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower()=='.wav'):
        x,sr,ch,key=read_audio(path)
        if key in seen:
            previous=seen[key]
            assert previous['label']==label,'Duplicado con etiquetas contradictorias'
            duplicates.append(dict(excluded=str(path),retained=previous['path']))
            continue
        rec=dict(path=str(path),name=path.name,kind=folder,label=label,sr=sr,channels=ch,
                 seconds=len(x)/sr,sha256=key[-1],saturation_pct=float(100*np.mean((x<=-1)|(x>=32767/32768))))
        rec['id']=len(records);records.append(rec);seen[key]=rec
assert records and len({r['sr'] for r in records})==1,'Vacío o distintas frecuencias: revisar datos'
SR=records[0]['sr']
for folder,label in FOLDERS:
    subset=[r for r in records if r['kind']==folder]
    print(folder,len(subset),'archivos únicos;',round(sum(r['seconds'] for r in subset),2),'s')
print('Duplicados excluidos:',duplicates)


## 2. Espectro, perfil de fondo y filtro

STFT con Hann de 32 ms, avance de 10 ms, FFT de la siguiente potencia de dos y normalización por energía de la ventana. No se aplica preénfasis ni normalización de amplitud por clip.

Para cada negativo de entrenamiento se obtiene la mediana de potencia por frecuencia sobre tramas muestreadas. Los perfiles de los archivos se promedian dando igual peso a cada archivo. Esto reduce la influencia de eventos breves y evita que una grabación larga domine; no elimina diferencias de micrófono o ambiente.

Para potencia observada P y perfil N: `S=max(P−αN,0)`, `G=max(piso,S/(S+αN+ε))`, `P_filtrada=G²P`. G es una ganancia de amplitud. El piso 0,20 permite hasta aproximadamente −14 dB de atenuación. La salida usada por el modelo es la potencia filtrada; no se reconstruye audio ni se afirma una mejora del SNR medido.


In [ ]:
FRAME=round(.032*SR);HOP=round(.010*SR);NFFT=2**int(np.ceil(np.log2(FRAME)))
WIN=np.hanning(FRAME)
freqs=np.fft.rfftfreq(NFFT,1/SR)
hz_to_mel=lambda hz:2595*np.log10(1+hz/700)
mel_to_hz=lambda mel:700*(10**(mel/2595)-1)
edges=mel_to_hz(np.linspace(hz_to_mel(0),hz_to_mel(SR/2),66))
MEL=np.maximum(0,np.minimum((freqs[None,:]-edges[:-2,None])/(edges[1:-1]-edges[:-2])[:,None],
                          (edges[2:,None]-freqs[None,:])/(edges[2:]-edges[1:-1])[:,None]))
assert np.all(MEL.sum(axis=1)>0)
MEL/=MEL.sum(axis=1,keepdims=True)
k=np.arange(20)[:,None];j=np.arange(64)[None,:]
DCT=np.sqrt(2/64)*np.cos(np.pi/64*(j+.5)*k);DCT[0]/=np.sqrt(2)

def power(clip):
    assert len(clip)>=FRAME
    frames=np.lib.stride_tricks.sliding_window_view(clip,FRAME)[::HOP]
    return np.abs(np.fft.rfft(frames*WIN,n=NFFT,axis=1))**2/np.sum(WIN**2)

def attenuate(P,N,alpha=ALPHA,floor=GAIN_FLOOR):
    assert P.shape[1]==len(N)
    noise=alpha*N[None,:]
    useful=np.maximum(P-noise,0)
    gain=np.maximum(floor,useful/(useful+noise+1e-12))
    return P*gain**2

def represent(P):
    logmel=10*np.log10(np.maximum(MEL@P.T,1e-12))
    mfcc=(DCT@logmel)[1:]
    return np.concatenate([mfcc.mean(axis=1),mfcc.std(axis=1)])

profile_by_file={};raw_vectors=[];rows=[]
for rec in records:
    x,_,_,_=read_audio(Path(rec['path']));size=round(WINDOW_S*SR)
    for start in range(0,len(x)-size+1,size):
        P=power(x[start:start+size]);raw_vectors.append(represent(P))
        rows.append(dict(file_id=rec['id'],file=rec['name'],kind=rec['kind'],label=rec['label'],
                         start_s=start/SR,end_s=(start+size)/SR))
    if rec['label']==0:
        chunks=[power(x[s:s+size]) for s in range(0,len(x)-size+1,round(PROFILE_STEP_S*SR))]
        assert chunks,'Negativo demasiado corto'
        profile_by_file[rec['id']]=np.median(np.concatenate(chunks,axis=0),axis=0)
X_raw=np.stack(raw_vectors);y=np.array([r['label'] for r in rows]);groups=np.array([r['file_id'] for r in rows])
assert np.isfinite(X_raw).all()
print('Ventanas y características:',X_raw.shape,'negativas:',int(sum(y==0)),'positivas:',int(sum(y==1)))
# Perfiles individuales precalculados: solo los IDs de entrenamiento podrán agregarse en cada fold.


## 3. Particiones por grabación
Se distribuyen los archivos únicos de cada clase entre particiones con semilla fija, garantizando ambas clases en cada evaluación. Con cuatro negativos se usan cuatro particiones. El mismo archivo nunca aparece en entrenamiento y evaluación simultáneamente. Esto no implica independencia por día o sesión: hay fechas compartidas y pocos negativos. No se ajustan parámetros con estas particiones.


In [ ]:
rng=np.random.default_rng(SEED)
by_class={lab:np.array([r['id'] for r in records if r['label']==lab]) for lab in [0,1]}
N_SPLITS=min(4,min(len(ids) for ids in by_class.values()))
assert N_SPLITS>=2
assignment={}
for label,ids in by_class.items():
    ids=rng.permutation(ids)
    for i,file_id in enumerate(ids):assignment[int(file_id)]=i%N_SPLITS
fold_of=np.array([assignment[int(g)] for g in groups])
splits=[]
for fold in range(N_SPLITS):
    tr=np.flatnonzero(fold_of!=fold);te=np.flatnonzero(fold_of==fold)
    assert not set(groups[tr])&set(groups[te])
    assert len(set(y[tr]))==len(set(y[te]))==2
    splits.append((tr,te))
    print('Fold',fold,'evaluación:',[r['name'] for r in records if assignment[r['id']]==fold])


In [ ]:
import matplotlib.pyplot as plt
import sklearn,joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,ConfusionMatrixDisplay

def weights(labels,ids):
    result=np.zeros(len(labels))
    for lab in [0,1]:
        unique=np.unique(ids[labels==lab])
        for g in unique:
            mask=(labels==lab)&(ids==g)
            result[mask]=1/(len(unique)*mask.sum())
    return result*len(result)/result.sum()

def metrics(labels,pred):
    tn,fp,fn,tp=confusion_matrix(labels,pred,labels=[0,1]).ravel()
    recall=tp/(tp+fn) if tp+fn else float('nan')
    specificity=tn/(tn+fp) if tn+fp else float('nan')
    return dict(VN=int(tn),FP=int(fp),FN=int(fn),VP=int(tp),
                precision=precision_score(labels,pred,zero_division=0),recall=recall,
                f1=f1_score(labels,pred,zero_division=0),specificity=specificity,
                balanced_accuracy=(recall+specificity)/2)

def filtered_features(noise):
    result=[]
    for rec in records:
        x,_,_,_=read_audio(Path(rec['path']));size=round(WINDOW_S*SR)
        for start in range(0,len(x)-size+1,size):
            result.append(represent(attenuate(power(x[start:start+size]),noise)))
    result=np.stack(result)
    assert result.shape==X_raw.shape and np.isfinite(result).all()
    return result

oof={name:dict(pred=np.full(len(y),-1),score=np.full(len(y),np.nan)) for name in ['MFCC_sin_filtro','MFCC_filtro_suave']}
fold_results=[];noise_profiles={};provenance=[];fold_models={}
for fold,(tr,te) in enumerate(splits):
    noise_ids=sorted(set(groups[tr][y[tr]==0]))
    assert not set(noise_ids)&set(groups[te])
    noise=np.mean([profile_by_file[g] for g in noise_ids],axis=0)
    noise_profiles[fold]=noise
    provenance.append(dict(fold=fold,noise_file_ids=[int(g) for g in noise_ids],
        noise_files=[records[g]['path'] for g in noise_ids],test_file_ids=sorted(int(g) for g in set(groups[te]))))
    X_filtered=filtered_features(noise)
    for name,X in [('MFCC_sin_filtro',X_raw),('MFCC_filtro_suave',X_filtered)]:
        model=make_pipeline(StandardScaler(),SVC(C=1.,kernel='rbf',gamma='scale'))
        model.fit(X[tr],y[tr],svc__sample_weight=weights(y[tr],groups[tr]))
        fold_models[(fold,name)]=model
        scores=model.decision_function(X[te]);pred=(scores>=0).astype(int)
        oof[name]['score'][te]=scores;oof[name]['pred'][te]=pred
        entry=dict(model=name,fold=fold,**metrics(y[te],pred));fold_results.append(entry)
        print(entry)
for result in oof.values():assert np.isfinite(result['score']).all() and np.all(result['pred']>=0)
summary=[dict(model=name,**metrics(y,r['pred'])) for name,r in oof.items()]
summary.append(dict(model='Siempre_dron',**metrics(y,np.ones_like(y))))
print('RESUMEN GLOBAL (conteos agrupados, no promedio de porcentajes):')
for row in summary:print(row)


## 4. Resultados y verificación del filtro
Precisión = VP/(VP+FP), recall = VP/(VP+FN), F1 = 2VP/(2VP+FP+FN). También se muestran especificidad y exactitud balanceada porque abundan los positivos. Menos FP junto con una caída grande de recall no demuestra una mejora general. La comparación es exploratoria: elegir parámetros a partir de estos resultados exige reservar nuevos datos externos para confirmarlos.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
for ax,(name,result) in zip(axes,oof.items()):
    ConfusionMatrixDisplay.from_predictions(y,result['pred'],labels=[0,1],display_labels=['Sin dron','Dron'],ax=ax,colorbar=False)
    ax.set(title=name+'\n[[VN, FP], [FN, VP]]',xlabel='Predicción',ylabel='Realidad')
plt.show()
per_file=[]
for rec in records:
    idx=np.flatnonzero(groups==rec['id'])
    for name,result in oof.items():
        entry=dict(file=rec['name'],kind=rec['kind'],fold=assignment[rec['id']],model=name,**metrics(y[idx],result['pred'][idx]))
        per_file.append(entry);print(entry)

# Inspección de un positivo EVALUADO con su perfil de entrenamiento correspondiente.
example=next(r for r in records if r['label']==1)
fold=assignment[example['id']];x,_,_,_=read_audio(Path(example['path']))
P=power(x[:round(WINDOW_S*SR)]);Q=attenuate(P,noise_profiles[fold])
raw_mel=10*np.log10(np.maximum(MEL@P.T,1e-12));clean_mel=10*np.log10(np.maximum(MEL@Q.T,1e-12))
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax,A,title in zip(axes,[raw_mel,clean_mel],['Original','Atenuación suave']):
    im=ax.imshow(A,origin='lower',aspect='auto',extent=[0,WINDOW_S,0,64],vmin=min(raw_mel.min(),clean_mel.min()),vmax=raw_mel.max(),cmap='magma')
    ax.set(title=title,xlabel='Tiempo aprox. (s)',ylabel='Banda Mel')
fig.colorbar(im,ax=axes,label='Potencia relativa (dB)');plt.show()
print('Archivo visualizado:',example['name'],'perfil del fold:',fold)


## Nueva prueba de robustez: cinco niveles de perturbación

Se añade **ruido blanco gaussiano sintético** a las ventanas de evaluación, a **20, 10, 5, 0 y −5 dB**, además del audio original como referencia. Menor valor significa más ruido añadido. Se usan tres realizaciones reproducibles por ventana; puedes reducirlas a una para una prueba rápida.

Estos dB expresan **potencia del audio original / potencia del ruido añadido**. El original ya contiene ruido ambiente, por lo que no equivalen al SNR aislado del dron. A 0 dB ambas potencias son iguales; a −5 dB el ruido añadido tiene más potencia que el original.

**Protocolo:** se mantienen los modelos de cada fold entrenados con audio original; solo se perturba su evaluación. El perfil del filtro permanece congelado y procede de negativos originales de entrenamiento. No hay reentrenamiento, adaptación al test ni selección de parámetros con estas pruebas. Las mismas realizaciones de ruido se aplican a las dos variantes y a todos los niveles para una comparación emparejada.

Las etiquetas de presencia se mantienen, incluso si el dron queda acústicamente enmascarado. Se añade ruido a positivos y negativos. Estas versiones son pruebas de estrés correlacionadas, **no nuevas grabaciones independientes ni aumento de datos de entrenamiento**. El ruido blanco no reproduce por sí solo viento, tráfico, lluvia o interferencias tonales.


In [ ]:
NOISE_LEVELS_DB=[20.,10.,5.,0.,-5.]
NOISE_REPEATS=3
PERTURB_SEED=20261006
assert NOISE_REPEATS>=1 and len(NOISE_LEVELS_DB)==5 and len(set(NOISE_LEVELS_DB))==5

def make_noise(length,file_id,start_sample,repeat):
    rng=np.random.default_rng(np.random.SeedSequence([PERTURB_SEED,int(file_id),int(start_sample),int(repeat)]))
    z=rng.standard_normal(length)
    z-=z.mean()
    return z/np.sqrt(np.mean(z*z))

def add_perturbation(x,z,level_db):
    original_power=float(np.mean(x*x))
    if original_power<=1e-20:
        raise ValueError('Ventana silenciosa: relación de potencias indefinida. Revisar antes de evaluar.')
    target_noise_power=original_power*10**(-level_db/10)
    noise=z*np.sqrt(target_noise_power/np.mean(z*z))
    mixed=x+noise
    actual_db=10*np.log10(original_power/np.mean(noise*noise))
    assert abs(actual_db-level_db)<1e-6
    # Trabajamos en float: no recortamos ni renormalizamos, para no introducir
    # distorsión adicional o cambiar el experimento. No guardar directamente como PCM16.
    return mixed,float(actual_db),float(np.mean(np.abs(mixed)>1))


In [ ]:
# Reutiliza fold_models entrenados en la sección 3. No usa modelos finales entrenados con todo.
assert len(fold_models)==N_SPLITS*2
perturbation_rows=[]
size=round(WINDOW_S*SR)
for rec in records:
    x,_,_,_=read_audio(Path(rec['path']))
    fold=assignment[rec['id']]
    assert rec['id'] not in provenance[fold]['noise_file_ids']
    current_indices=np.flatnonzero(groups==rec['id'])
    for repeat in range(NOISE_REPEATS):
        batches={level:{'raw':[],'filtered':[],'meta':[]} for level in NOISE_LEVELS_DB}
        for i in current_indices:
            row=rows[i];start=round(row['start_s']*SR);clip=x[start:start+size]
            z=make_noise(len(clip),rec['id'],start,repeat)
            for level in NOISE_LEVELS_DB:
                mixed,measured,over=add_perturbation(clip,z,level)
                P=power(mixed)
                batches[level]['raw'].append(represent(P))
                batches[level]['filtered'].append(represent(attenuate(P,noise_profiles[fold])))
                batches[level]['meta'].append(dict(window_index=int(i),file=rec['name'],kind=rec['kind'],fold=fold,
                    start_s=row['start_s'],end_s=row['end_s'],label=int(y[i]),repeat=repeat,
                    level_db=level,measured_db=measured,fraction_above_full_scale=over))
        for level,batch in batches.items():
            for name,key in [('MFCC_sin_filtro','raw'),('MFCC_filtro_suave','filtered')]:
                scores=fold_models[(fold,name)].decision_function(np.stack(batch[key]))
                for meta,score in zip(batch['meta'],scores):
                    perturbation_rows.append(dict(meta,model=name,score=float(score),prediction=int(score>=0)))
    print('Perturbaciones completadas:',rec['name'])
expected=len(rows)*len(NOISE_LEVELS_DB)*NOISE_REPEATS*2
assert len(perturbation_rows)==expected
print('Predicciones perturbadas:',len(perturbation_rows))


### Métricas por nivel y repetición
Cada combinación nivel/repetición contiene exactamente las mismas ventanas originales fuera de entrenamiento. Se calcula VN, FP, FN, VP, precisión, recall, F1, especificidad y exactitud balanceada. El resumen informa media y desviación entre realizaciones de ruido: **no son intervalos de confianza ni variabilidad entre sesiones**. Los resultados sin perturbación se añaden una sola vez.


In [ ]:
perturb_metrics=[];perturb_fold_metrics=[]
for name in oof:
    perturb_metrics.append(dict(model=name,condition='original',level_db='',repeat=-1,**metrics(y,oof[name]['pred'])))
    for level in NOISE_LEVELS_DB:
        for repeat in range(NOISE_REPEATS):
            subset=[r for r in perturbation_rows if r['model']==name and r['level_db']==level and r['repeat']==repeat]
            assert len(subset)==len(y) and len({r['window_index'] for r in subset})==len(y)
            labels=np.array([r['label'] for r in subset]);pred=np.array([r['prediction'] for r in subset])
            result=dict(model=name,condition='gaussian',level_db=level,repeat=repeat,**metrics(labels,pred))
            perturb_metrics.append(result);print(result)
            for fold in range(N_SPLITS):
                part=[r for r in subset if r['fold']==fold]
                perturb_fold_metrics.append(dict(model=name,level_db=level,repeat=repeat,fold=fold,
                    **metrics(np.array([r['label'] for r in part]),np.array([r['prediction'] for r in part]))))

metric_names=['precision','recall','f1','specificity','balanced_accuracy','VN','FP','FN','VP']
robustness_summary=[]
for name in oof:
    for level in [None]+NOISE_LEVELS_DB:
        subset=[r for r in perturb_metrics if r['model']==name and (r['condition']=='original' if level is None else r['condition']=='gaussian' and r['level_db']==level)]
        result=dict(model=name,condition='original' if level is None else 'gaussian',level_db='' if level is None else level)
        for key in metric_names:
            values=np.array([r[key] for r in subset],float)
            result[key+'_mean']=float(np.mean(values));result[key+'_std']=float(np.std(values))
        robustness_summary.append(result)
print('RESUMEN: media por realización de ruido')
for r in robustness_summary:
    print(r['model'],r['condition'],r['level_db'],'precisión',round(r['precision_mean'],4),
          'recall',round(r['recall_mean'],4),'F1',round(r['f1_mean'],4),
          'FP medios',round(r['FP_mean'],2),'FN medios',round(r['FN_mean'],2))


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(15,8),constrained_layout=True)
labels=['Original']+[f'{v:g} dB' for v in NOISE_LEVELS_DB]
for ax,key,title in zip(axes.ravel(),['precision','recall','f1','specificity','FP','FN'],
                       ['Precisión','Recall','F1','Especificidad','Falsos positivos','Falsos negativos']):
    for name in oof:
        selected=[r for r in robustness_summary if r['model']==name]
        means=np.array([r[key+'_mean'] for r in selected]);std=np.array([r[key+'_std'] for r in selected])
        ax.errorbar(range(len(labels)),means,yerr=std,marker='o',capsize=3,label=name)
    ax.set_title(title);ax.set_xticks(range(len(labels)),labels,rotation=25);ax.grid(alpha=.2)
    if key not in ['FP','FN']:ax.set_ylim(0,1.05)
axes[0,0].legend(fontsize=8)
fig.suptitle('Robustez al ruido añadido: entrenamiento y filtro congelados')
plt.show()

# Conteos enteros de la primera realización; no mezclar con las medias de las curvas.
fig,axes=plt.subplots(2,6,figsize=(20,7),constrained_layout=True)
for row,name in enumerate(oof):
    for col,level in enumerate([None]+NOISE_LEVELS_DB):
        if level is None:truth=y;prediction=oof[name]['pred']
        else:
            part=[r for r in perturbation_rows if r['model']==name and r['level_db']==level and r['repeat']==0]
            truth=np.array([r['label'] for r in part]);prediction=np.array([r['prediction'] for r in part])
        ConfusionMatrixDisplay.from_predictions(truth,prediction,labels=[0,1],display_labels=['Sin','Dron'],ax=axes[row,col],colorbar=False)
        axes[row,col].set_title(name+'\n'+('Original' if level is None else f'{level:g} dB, repetición 0'),fontsize=8)
plt.show()


In [ ]:
OUT.mkdir(exist_ok=True)
for name,data in [('perturbaciones_predicciones',perturbation_rows),('perturbaciones_metricas',perturb_metrics),
                  ('perturbaciones_folds',perturb_fold_metrics),('perturbaciones_resumen',robustness_summary)]:
    with (OUT/f'{name}.csv').open('w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=list(data[0]));writer.writeheader();writer.writerows(data)
perturb_config=dict(noise='additive_white_gaussian',levels_db=NOISE_LEVELS_DB,repeats=NOISE_REPEATS,
    seed=PERTURB_SEED,reference='original recording power, not isolated drone power',
    clipping=False,renormalization=False,training_augmented=False,paired_noise_across_models_and_levels=True,
    noise_profiles_frozen=True,folds=provenance)
(OUT/'perturbaciones_configuracion.json').write_text(json.dumps(perturb_config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Guardadas las métricas, predicciones y condiciones de las perturbaciones en',OUT)


## 5. Guardar evaluación y procedencia
Las predicciones son fuera de entrenamiento. Se guardan también los negativos usados para el perfil de cada partición. Las métricas por archivo de una sola clase pueden contener NaN donde no están definidas.


In [ ]:
OUT.mkdir(exist_ok=True)
predictions=[dict(row,fold=int(fold_of[i]),raw_prediction=int(oof['MFCC_sin_filtro']['pred'][i]),
                 filtered_prediction=int(oof['MFCC_filtro_suave']['pred'][i]),
                 raw_margin=float(oof['MFCC_sin_filtro']['score'][i]),filtered_margin=float(oof['MFCC_filtro_suave']['score'][i])) for i,row in enumerate(rows)]
for name,data in [('inventario',records),('resumen',summary),('metricas_folds',fold_results),('metricas_archivos',per_file),('predicciones_oof',predictions)]:
    with (OUT/f'{name}.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=list(data[0]));w.writeheader();w.writerows(data)
config=dict(sr=SR,channel=CHANNEL,window_s=WINDOW_S,alpha=ALPHA,gain_floor=GAIN_FLOOR,
            profile_step_s=PROFILE_STEP_S,seed=SEED,duplicates=duplicates,noise_provenance=provenance,
            sklearn=sklearn.__version__,numpy=np.__version__,python=platform.python_version())
(OUT/'configuracion.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
np.savez_compressed(OUT/'perfiles_por_fold.npz',**{f'fold_{k}':v for k,v in noise_profiles.items()})
print('Resultados:',OUT)


## 6. Modelos finales e inferencia opcional
Se entrenan ambos prototipos con todos los archivos únicos. El perfil final utiliza todos los negativos disponibles; esto es válido para entrenar un modelo destinado a audios futuros, pero sus predicciones en los archivos actuales no son evaluación independiente. Se guardan los parámetros del frontend junto a los modelos. El notebook contiene el código necesario para reproducirlo. Joblib debe cargarse solo desde fuentes de confianza.


In [ ]:
TRAIN_FINAL=True
if TRAIN_FINAL:
    final_noise=np.mean(list(profile_by_file.values()),axis=0)
    final_filtered=filtered_features(final_noise)
    final_models={}
    for name,X in [('raw',X_raw),('filtered',final_filtered)]:
        m=make_pipeline(StandardScaler(),SVC(C=1.,kernel='rbf',gamma='scale'))
        m.fit(X,y,svc__sample_weight=weights(y,groups));final_models[name]=m
    bundle=dict(models=final_models,noise_profile=final_noise,config=config,
                mel=MEL,dct=DCT,window=WIN,frame=FRAME,hop=HOP,nfft=NFFT,
                training_hashes=[r['sha256'] for r in records])
    joblib.dump(bundle,OUT/'modelos_y_perfil.joblib')
    print('Guardados los dos modelos y el perfil final.')

NEW_AUDIO=None  # Path(r'C:\ruta\audio_nuevo.wav')
if NEW_AUDIO is not None:
    bundle=joblib.load(OUT/'modelos_y_perfil.joblib')
    cfg=bundle['config']
    assert (SR,CHANNEL,WINDOW_S,ALPHA,GAIN_FLOOR)==(cfg['sr'],cfg['channel'],cfg['window_s'],cfg['alpha'],cfg['gain_floor']), 'Usa la configuración con que entrenaste.'
    x,sr,_,key=read_audio(Path(NEW_AUDIO));assert sr==SR,'Se necesita la misma frecuencia de muestreo'
    if key[-1] in bundle['training_hashes']:print('Advertencia: este audio fue usado para entrenamiento.')
    size=round(WINDOW_S*SR);results=[]
    for start in range(0,len(x)-size+1,size):
        P=power(x[start:start+size]);Q=attenuate(P,bundle['noise_profile'])
        entry=dict(start_s=start/SR,end_s=(start+size)/SR)
        for name,A in [('raw',P),('filtered',Q)]:
            score=float(bundle['models'][name].decision_function(represent(A)[None,:])[0])
            entry[name+'_margin']=score;entry[name+'_prediction']=int(score>=0)
        results.append(entry)
    assert results,'Audio demasiado corto'
    with (OUT/'predicciones_audio_nuevo.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=list(results[0]));w.writeheader();w.writerows(results)
    print(results[:10])


## Límites y próximos experimentos
- El perfil puede no representar un ambiente nuevo. Una disminución del ruido percibido no prueba mejores métricas.
- No hay un filtro distinto elegido según la etiqueta de evaluación; ambas clases pasan por el mismo procesamiento.
- El piso evita borrado completo, pero no garantiza conservar armónicos débiles. No se ajustó con el conjunto evaluado.
- Conviene corroborar por fecha/sesión y por tipo de dron. No extraigas conclusiones estadísticas de miles de ventanas como si fueran grabaciones independientes.
- La presente comparación mantiene MFCC y clasificador fijos. PCEN y log-Mel como entradas directas quedan para experimentos separados.
- No compara contra los valores antiguos como si fueran el mismo conjunto: ahora se usan Hexa, Quad y cuatro negativos únicos.
